In [1]:
import os
import pickle
import numpy as np
from tqdm import tqdm

In [2]:
dataset = "LastFM"
head_ratio = 0.2
user_emb = pickle.load(open(os.path.join(dataset+"/handled/", "usr_emb_np.pkl"), "rb"))

In [3]:
user_emb[0]

array([0., 0., 0., ..., 0., 0., 0.])

In [4]:
user_len = {}
flag = 0
with open(os.path.join(dataset+"/handled/inter2seq.txt"), 'r') as f:
    for line in tqdm(f.readlines()):
        parts = line.strip().split()
        user_id = int(parts[0])
        seq_len = len(parts) - 1
        user_len[user_id] = seq_len

100%|██████████| 1390/1390 [00:00<00:00, 255570.86it/s]


In [5]:
sorted_users = sorted(user_len.items(), key=lambda x: x[1], reverse=True)
num_users = len(sorted_users)
head_user_num = int(num_users * head_ratio)
print(f"Total users: {num_users}, Head users: {head_user_num}, Tail users: {num_users - head_user_num}")
head_users = [u[0] for u in sorted_users[:head_user_num]]
tail_users = [u[0] for u in sorted_users[head_user_num:]]

Total users: 1390, Head users: 278, Tail users: 1112


In [6]:
head_user_strs = [str(u) + '\n' for u in head_users]
tail_user_strs = [str(u) + '\n' for u in tail_users]
with open(os.path.join(dataset+"/handled/", "head_user_id.txt"), 'w') as f:
    f.writelines(head_user_strs)
with open(os.path.join(dataset+"/handled/", "tail_user_id.txt"), 'w') as f:
    f.writelines(tail_user_strs)

In [7]:
uid2idx = {}
seen = set()
ordered = []
with open(os.path.join(dataset, "handled", "inter.txt"), "r") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        u = int(parts[0])
        if u not in seen:
            seen.add(u)
            ordered.append(u)
uid2idx = {u: idx for idx, u in enumerate(ordered)}
valid_indices = []
miss = 0
for u in head_users:
    idx = uid2idx.get(u, None)
    if idx is not None and idx < len(user_emb):
        valid_indices.append(idx)
    else:
        miss += 1
if miss > 0:
    print(f"Skip {miss} head users without aligned embedding")
head_user_emb = user_emb[valid_indices]
print(f"Head user embedding array shape: {head_user_emb.shape}")
pickle.dump(head_user_emb, open(os.path.join(dataset, "handled", "head_usr_emb_np.pkl"), "wb"))

Head user embedding array shape: (278, 1536)


In [8]:
head_user_emb[0]

array([0., 0., 0., ..., 0., 0., 0.])

In [9]:
uid2idx = {}
seen = set()
ordered = []
with open(os.path.join(dataset, "handled", "inter.txt"), "r") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        u = int(parts[0])
        if u not in seen:
            seen.add(u)
            ordered.append(u)
uid2idx = {u: idx for idx, u in enumerate(ordered)}
valid_indices = []
miss = 0
for u in tail_users:
    idx = uid2idx.get(u, None)
    if idx is not None and idx < len(user_emb):
        valid_indices.append(idx)
    else:
        miss += 1
if miss > 0:
    print(f"Skip {miss} tail users without aligned embedding")
tail_user_emb = user_emb[valid_indices]
print(f"Tail user embedding array shape: {tail_user_emb.shape}")
pickle.dump(tail_user_emb, open(os.path.join(dataset, "handled", "tail_usr_emb_np.pkl"), "wb"))

Tail user embedding array shape: (1112, 1536)
